# Fine-Tuning Indic-Speak on Marathi with LoRA

This notebook runs the complete Marathi TTS fine-tuning and inference pipeline on a Kaggle NVIDIA T4 GPU.
- **Base Model**: `bodhan-ai/indic-speak` (3.3B parameters, SNAC 24kHz codec)
- **Vocoder**: Fine-tuned Vocos 24kHz neural decoder
- **Dataset**: `snorbyte/indic-tts-sample-snac-encoded` (filtered Marathi subset)
- **Precision**: `fp16` (matched to Kaggle T4 GPU)

In [1]:
# 1. Clone repository and install dependencies
import os
repo_dir = "/kaggle/working/indic-speak-marathi-finetune"
if not os.path.exists(repo_dir):
    !git clone https://github.com/mehersoni/indic-speak-marathi-finetune.git {repo_dir}
%cd {repo_dir}
!git pull

# Remove conflicting pre-installed torchao on Kaggle and install dependencies
!pip uninstall -y torchao
!pip install -q "transformers>=5" peft accelerate snac soundfile datasets pandas pyarrow huggingface_hub pyyaml

Cloning into '/kaggle/working/indic-speak-marathi-finetune'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.34)
remote: Compressing objects: 100% (79/79), done.
remote: Total 134 (delta 63), reused 118 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 37.82 KiB | 2.91 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/kaggle/working/indic-speak-marathi-finetune
Already up to date.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
# 2. Hugging Face Authentication
import os
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")

if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face authenticated successfully.")
else:
    print("HF_TOKEN not detected in Kaggle Secrets. If needed, set os.environ['HF_TOKEN'] = 'your_token'.")

Hugging Face authenticated successfully.


In [3]:
# 3. Prepare and filter Marathi dataset (Threshold: 1400 tokens)
!CUDA_VISIBLE_DEVICES=0 python scripts/prepare_dataset.py

data_stage_1.parquet: 100%|██████████████████| 125M/125M [00:02<00:00, 61.0MB/s]
Dataset path: /root/indic-dataset-cache/data_stage_1.parquet
Total rows in dataset: 63281
Columns: ['utterance', 'language', 'emotion', 'type', 'act', 'rating', 'gender', 'age', 'environment', 'user', 'snac_codes', 'stage']

--- Marathi Subset Statistics ---
Initial Marathi row count: 7604
tokenizer_config.json: 100%|███████████████████| 373/373 [00:00<00:00, 2.39MB/s]
tokenizer.json: 100%|██████████████████████| 22.7M/22.7M [00:00<00:00, 27.9MB/s]

--- Sequence Length Filtering (Threshold: 1400 tokens) ---
Rows dropped (seq_len > 1400): 73 (0.96%)
Rows remaining (seq_len <= 1400): 7531 (99.04%)

Filtered Marathi Speaker Count: 9
Filtered Utterances per speaker:
user
44     4468
82     1406
66      545
55      422
80      389
148     194
147      61
67       41
43        5
Name: count, dtype: int64

Filtered Sequence Length Distribution:
  Count:  7531
  Min:    91
  Max:    1392
  Mean:   695.9
  Median: 

In [4]:
# 4. Run GPU Smoke Test (Forward, Backward, Optimizer, NaN/Inf gradient checks)
!CUDA_VISIBLE_DEVICES=0 python scripts/smoke_test.py --config configs/marathi_lora.yaml

--- Environment ---
Device: cuda
Precision: torch.float16
Loading model/tokenizer from: bodhan-ai/indic-speak
model.safetensors: 100%|████████████████████| 6.60G/6.60G [00:26<00:00, 254MB/s]
Loading weights: 100%|█| 254/254 [00:03<00:00, 71.16it/s, Materializing param=mo
generation_config.json: 100%|███████████████████| 205/205 [00:00<00:00, 706kB/s]

[Stage 1] Single Example Forward Pass:
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
  PASS: Loss is finite float (4.3637)

[Stage 2] Single Example Forward + Backward + Optimizer Step:
  PASS: Gradients are finite; optimizer step completed successfully

[Stage 3] Batched (2 samples) Forward Pass:
  PASS: Batch loss is finite float (4.5593)

[Stage 4] Batched (2 samples) Forward + Backward + Optimizer Step:
  PASS: Batch gradients are finite; optimizer step completed successfully

ALL SMOKE TEST STAGES PASSED SUCCESSFULLY


In [5]:
# 5. Train LoRA adapter on Marathi speech dataset
!CUDA_VISIBLE_DEVICES=0 python src/train.py --config configs/marathi_lora.yaml

Loading tokenizer and model from: bodhan-ai/indic-speak
Loading weights: 100%|█| 254/254 [00:03<00:00, 73.52it/s, Materializing param=mo

Identified linear attention target modules: ['down_proj', 'gate_proj', 'k_proj', 'o_proj', 'q_proj', 'up_proj', 'v_proj']

Trainable parameters: 24,313,856 / 3,325,242,368 (0.7312%)

Loading dataset from: data_stage_1.parquet
Dataset split: 3500 train, 100 validation samples

Starting LoRA fine-tuning...
{'loss': '4.376', 'grad_norm': '0.4424', 'learning_rate': '5.4e-06', 'epoch': '0.02286'}
{'loss': '4.392', 'grad_norm': '0.3818', 'learning_rate': '1.14e-05', 'epoch': '0.04571'}
{'loss': '4.334', 'grad_norm': '0.409', 'learning_rate': '1.74e-05', 'epoch': '0.06857'}
{'loss': '4.213', 'grad_norm': '0.371', 'learning_rate': '2.34e-05', 'epoch': '0.09143'}
{'loss': '4.047', 'grad_norm': '0.2326', 'learning_rate': '2.94e-05', 'epoch': '0.1143'}
 99%|█████████████████████████████████████████▌| 99/100 [00:46<00:00,  1.83it/s]
                             

In [6]:
# 6. Generate speech samples with Base Model
!CUDA_VISIBLE_DEVICES=0 python src/inference.py --model bodhan-ai/indic-speak --out-dir outputs/examples

Loading base model and tokenizer from: bodhan-ai/indic-speak
Loading weights: 100%|█| 254/254 [00:03<00:00, 67.41it/s, Materializing param=mo
Loading SNAC 24kHz codec and Vocos decoder...
config.json: 100%|██████████████████████████████| 300/300 [00:00<00:00, 938kB/s]
pytorch_model.bin: 100%|███████████████████| 79.5M/79.5M [00:01<00:00, 55.7MB/s]
vocos/best.pt: 100%|██████████████████████████| 467M/467M [00:03<00:00, 122MB/s]

Generating audio for 4 sentences (Prefix: base_NN.wav)...
  [base_00] 3.75s | 309 audio tokens (prompt: 39 tokens) -> outputs/examples/base_00.wav
       Text: "नमस्कार, आज आपण विज्ञान विषयाचा अभ्यास करणार आहोत." (50 chars, 8 words)

  [base_01] 3.93s | 323 audio tokens (prompt: 31 tokens) -> outputs/examples/base_01.wav
       Text: "मॅडम, काही मदत हवी आहे का?" (26 chars, 6 words)

  [base_02] 5.55s | 456 audio tokens (prompt: 43 tokens) -> outputs/examples/base_02.wav
       Text: "महाराष्ट्र हे भारतातील एक पुरोगामी आणि महत्त्वाचे राज्य आहे." (60 chars, 9 word

In [9]:
!git pull

remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 48 (delta 33), reused 32 (delta 17), pack-reused 0 (from 0)
Unpacking objects: 100% (48/48), 22.34 KiB | 618.00 KiB/s, done.
From https://github.com/mehersoni/indic-speak-marathi-finetune
   10451cc..9523563  main       -> origin/main
Updating 10451cc..9523563
Fast-forward
 configs/marathi_lora_full.yaml |  34 +++
 requirements.txt               |   1 +
 scripts/evaluate_asr.py        |  73 +++++++
 scripts/generate_report_doc.py | 479 +++++++++++++++++++++++++++++++++++++++++
 src/dataset.py                 |   7 +-
 src/inference.py               |   8 +-
 src/train.py                   |   4 +
 7 files changed, 601 insertions(+), 5 deletions(-)
 create mode 100644 configs/marathi_lora_full.yaml
 create mode 100644 scripts/evaluate_asr.py
 create mode 100644 scripts/generate_report_doc.py


In [10]:
# 7. Generate speech samples with Fine-Tuned LoRA Adapter
!CUDA_VISIBLE_DEVICES=0 python src/inference.py \
    --model bodhan-ai/indic-speak \
    --adapter outputs/lora_marathi/final_adapter \
    --out-dir outputs/examples

Loading base model and tokenizer from: bodhan-ai/indic-speak
Loading weights: 100%|█| 254/254 [00:03<00:00, 73.09it/s, Materializing param=mo
Loading LoRA adapter from: outputs/lora_marathi/final_adapter
Loading SNAC 24kHz codec and Vocos decoder...

Generating audio for 4 sentences (Prefix: finetuned_NN.wav)...
  [finetuned_00] 6.14s | 505 audio tokens (prompt: 39 tokens) -> outputs/examples/finetuned_00.wav
       Text: "नमस्कार, आज आपण विज्ञान विषयाचा अभ्यास करणार आहोत." (50 chars, 8 words)

  [finetuned_01] 3.50s | 288 audio tokens (prompt: 31 tokens) -> outputs/examples/finetuned_01.wav
       Text: "मॅडम, काही मदत हवी आहे का?" (26 chars, 6 words)

  [finetuned_02] 7.34s | 603 audio tokens (prompt: 43 tokens) -> outputs/examples/finetuned_02.wav
       Text: "महाराष्ट्र हे भारतातील एक पुरोगामी आणि महत्त्वाचे राज्य आहे." (60 chars, 9 words)

  [finetuned_03] 3.58s | 295 audio tokens (prompt: 30 tokens) -> outputs/examples/finetuned_03.wav
       Text: "शिक्षण हे मानवी जीवनाचा पाया 

In [11]:
# 8. Play Generated Audio Samples
import IPython.display as ipd
from pathlib import Path

audio_files = sorted(Path("outputs/examples").glob("*.wav"))
print(f"Found {len(audio_files)} audio files in outputs/examples:\n")
for f in audio_files:
    print(f"--- {f.name} ---")
    ipd.display(ipd.Audio(str(f)))

Found 8 audio files in outputs/examples:

--- base_00.wav ---


--- base_01.wav ---


--- base_02.wav ---


--- base_03.wav ---


--- finetuned_00.wav ---


--- finetuned_01.wav ---


--- finetuned_02.wav ---


--- finetuned_03.wav ---


In [12]:
# ── Submission Prep Cell ────────────────────────────────────────────────
import os, shutil, json
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

WORKING = "/kaggle/working"
OUT_DIR = f"{WORKING}/marathi_tts_submission"
os.makedirs(f"{OUT_DIR}/audio", exist_ok=True)
os.makedirs(f"{OUT_DIR}/model", exist_ok=True)
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)

# 1. Copy generated audio
for f in os.listdir("outputs/examples"):
    if f.endswith(".wav"):
        shutil.copy(f"outputs/examples/{f}", f"{OUT_DIR}/audio/{f}")

# 2. Copy LoRA adapter (the trained model)
ADAPTER = "outputs/lora_marathi/final_adapter"
if os.path.exists(ADAPTER):
    shutil.copytree(ADAPTER, f"{OUT_DIR}/model/final_adapter", dirs_exist_ok=True)

# 3. Copy trainer_state.json (has every step loss)
STATE = "outputs/lora_marathi/trainer_state.json"
if os.path.exists(STATE):
    shutil.copy(STATE, f"{OUT_DIR}/trainer_state.json")

# 4. Plot training loss from trainer_state.json
if os.path.exists(STATE):
    with open(STATE) as f:
        state = json.load(f)
    
    train_log = [(e["step"], e["loss"]) for e in state["log_history"] if "loss" in e]
    eval_log  = [(e["step"], e["eval_loss"]) for e in state["log_history"] if "eval_loss" in e]
    
    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)
    if train_log:
        steps, losses = zip(*train_log)
        ax.plot(steps, losses, label="Train Loss", color="#0284c7", alpha=0.8, lw=1.6)
    if eval_log:
        steps, losses = zip(*eval_log)
        ax.plot(steps, losses, label="Val Loss", color="#e11d48", lw=2, linestyle="--", marker="o", ms=4)
    
    ax.set_xlabel("Step"); ax.set_ylabel("Loss")
    ax.set_title("Marathi TTS LoRA — Training Convergence")
    ax.legend(); ax.grid(True, linestyle=":", alpha=0.5)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/figures/loss_curve.png")
    plt.close()
    print("✓ Loss curve saved")

# 5. Generate base model audio for comparison (no adapter)
print("\nGenerating BASE model audio for comparison...")
from src.inference import run_inference
run_inference(
    base_model_path="bodhan-ai/indic-speak",
    adapter_path=None,          # no LoRA — pure base
    output_dir=f"{OUT_DIR}/audio/base",
)

# 6. Print token/duration summary table
print("\n── Inference Summary ──")
print(f"{'File':<20} {'Duration':>10} {'Tokens':>8}")
print("-" * 42)
for f in sorted(os.listdir(f"{OUT_DIR}/audio")):
    if f.endswith(".wav"):
        import soundfile as sf
        wav, sr = sf.read(f"{OUT_DIR}/audio/{f}")
        dur = len(wav) / sr
        print(f"  {f:<18} {dur:>8.2f}s")

# 7. Zip everything
zip_path = f"{WORKING}/marathi_tts_submission"
shutil.make_archive(zip_path, "zip", OUT_DIR)
print(f"\n✓ ZIP ready: {zip_path}.zip")
print("  → Download from the Output tab (right sidebar)")


Generating BASE model audio for comparison...
Loading base model and tokenizer from: bodhan-ai/indic-speak


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading SNAC 24kHz codec and Vocos decoder...

Generating audio for 4 sentences (Prefix: base_NN.wav)...
  [base_00] 4.27s | 351 audio tokens (prompt: 39 tokens) -> /kaggle/working/marathi_tts_submission/audio/base/base_00.wav
       Text: "नमस्कार, आज आपण विज्ञान विषयाचा अभ्यास करणार आहोत." (50 chars, 8 words)

  [base_01] 2.82s | 232 audio tokens (prompt: 31 tokens) -> /kaggle/working/marathi_tts_submission/audio/base/base_01.wav
       Text: "मॅडम, काही मदत हवी आहे का?" (26 chars, 6 words)

  [base_02] 5.72s | 470 audio tokens (prompt: 43 tokens) -> /kaggle/working/marathi_tts_submission/audio/base/base_02.wav
       Text: "महाराष्ट्र हे भारतातील एक पुरोगामी आणि महत्त्वाचे राज्य आहे." (60 chars, 9 words)

  [base_03] 3.50s | 288 audio tokens (prompt: 30 tokens) -> /kaggle/working/marathi_tts_submission/audio/base/base_03.wav
       Text: "शिक्षण हे मानवी जीवनाचा पाया आहे." (33 chars, 6 words)


── Inference Summary ──
File                   Duration   Tokens
------------------------